# Price vs. salary: how many monthly salaries does 1 m² cost?

Compares the median apartment price per m² (from the cleaned listings, outliers excluded) with the average gross monthly salary in 15 Polish cities.

**Data**
- Salaries: GUS Local Data Bank API (BDL), variable 64428, average monthly gross wages by powiat, 2023 and 2024.
- Prices: `data/clean/apartments_clean.csv`, listings from Aug 2023 to Jun 2024.

**Steps**
1. Download all 380 powiats from the API (4 pages of 100).
2. Replace invalid zero salaries with NaN.
3. Keep only the cities that appear in the apartment data (powiats marked `m.`, plus `m. st. Warszawa`).
4. Join the salary table with the median price per m² by city.
5. Compute `months_per_m2` = median price per m² / 2024 salary.

**Limitations**
- Salaries are yearly averages; prices cover 11 months.
- The 2024 salary is used for the whole price period.
- Listing prices are asking prices, not closed sales.

In [1]:
import requests
import pandas as pd
import numpy as np
import os
from unidecode import unidecode

In [2]:
r = requests.get("https://bdl.stat.gov.pl/api/v1/data/by-variable/64428?format=json&unit-level=5&year=2023&year=2024")
print(r.status_code)
dados = r.json()
print(dados.keys())

200
dict_keys(['totalRecords', 'links', 'variableId', 'measureUnitId', 'aggregateId', 'lastUpdate', 'results'])


In [3]:
lista = []
df = []

for x in range(4):
    url = f"https://bdl.stat.gov.pl/api/v1/data/by-variable/64428?format=json&unit-level=5&year=2023&year=2024&page={x}&page-size=100"
    r = requests.get(url)
    dados = r.json()
    lista.append(url)
    for i in dados['results']:
        df.append({'id':i['id'],'city':i['name'],
                      'sal_2023':i['values'][0]['val'],'sal_2024':i['values'][1]['val']})

df = pd.DataFrame(df)
    

In [4]:
df['city'] = df['city'].str.replace('Powiat','').str.strip().str.lower()

df.sort_values(by='sal_2024', ascending=False).head(5)

df.loc[(df['sal_2023'] <= 0),'sal_2023'] = np.nan
df.loc[(df['sal_2024'] <= 0),'sal_2024'] = np.nan


In [5]:
apt_df = pd.read_csv('../data/clean/apartments_clean.csv')

apt = apt_df['city'].unique().tolist()

cities = {}
df['city'] = df['city'].apply(unidecode)

for i in apt:
    cities[f'm. {i}'] = i

cities['m. st. warszawa'] = 'warszawa'


In [6]:
df['city_key'] = df['city'].map(cities)
df.dropna(subset=['city_key'])
sal = df.dropna(subset=['city_key'])

In [7]:
precos = apt_df[apt_df['is_outlier'] == False].groupby('city')['price_per_m2'].median().sort_values(ascending=False).reset_index()
precos = precos.rename(columns={'price_per_m2':'median_price_m2'})


In [8]:
sal = sal.drop(columns=['city'])
sal = sal.rename(columns={'city_key':'city'})


In [9]:
total = sal.merge(precos, on='city',how='left')
total['months_per_m2'] = round(total['median_price_m2'] / total['sal_2024'],2)
total = total.sort_values(by='months_per_m2',ascending=False)
